In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.sports-commentary",
        description="Automatically generates factual, neutral play-by-play commentary for a video clip already identified as a sports highlight moment.",
        worker_release="qwen3-instruct",
        text_prompt="""
          You are a neutral sports commentator. Analyze the provided video clip, which has
          already been identified as containing a highlight moment.

          Describe what happens in the clip as factual, neutral play-by-play commentary,
          as if narrating for a sports broadcast recap.

          Include:
          - Which player or team is involved, identified only by position, jersey number,
          or jersey color if visible. Do not guess a specific player's name or identity.
          - The specific action that occurs (e.g. a shot, a tackle, a save, a goal).
          - The immediate, visible outcome of that action.

          Exclude:
          - Speculation about player intent or strategy.
          - Dramatic or emotional language.
          - Commentary on broader game context (score implications, momentum, storylines)
          unless directly visible in the clip.

          Write 1-2 sentences. Do not use markdown formatting, headers, or bullet points
          in your response.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            fps=5,
            image_size=640
        ),
        is_public=False
    )
]

### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Video

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.describe.sports-commentary:latest"
   )
])

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_vid_path = Path("/content/sample_vid.mp4") # change to the path of your sample video
   job = endpoint.upload(sample_vid_path)
   while result := job.predict():
     print(json.dumps(result, indent=2))

print("Done")